# 面试题：如何从零实现 DETR 的集合预测、匹配与目标检测？

## 面试回答主线

DETR 把目标检测写成固定数量 query 的集合预测：图像先变成带位置的视觉 token，query 对 token 做 cross-attention，再分别输出类别（含 `no-object`）和归一化框。训练时不能把“第一个标注”永久绑给“第一个 query”，而要用二分匹配寻找当前代价最小的一一对应；未匹配 query 监督为 `no-object`。损失通常同时包含分类、框坐标 L1 与 IoU/GIoU，推理时由 `no-object` 类自己决定输出数量，不能偷看真实目标数。

本实验只用 PyTorch 底层张量运算手写 patch 特征、query cross-attention、集合匹配和损失。特别注意：模型输入只来自 RGB 图像；真实类别和框严格只进入 matching/loss，不会拼进 candidate token。

## 真实案例：仓库俯视图中的纸箱与蓝色料桶

下面生成 18 张训练图和 6 张留出图。每张是 16×16 RGB 俯视网格，红色纹理区域表示纸箱、蓝色纹理区域表示料桶，背景含轻微传感器噪声。框和类别来自独立标注表，仅用于监督。数据是可视化的受控教学图，不代表真实相机、遮挡或跨仓泛化。

In [1]:
import itertools  # 导入排列工具以手写小规模二分匹配。
import math  # 导入平方根用于缩放点积注意力。
import torch  # 导入 PyTorch 以构造图像和真实反向传播。
from torch import nn  # 导入模块与可学习参数容器。
import torch.nn.functional as F  # 导入底层卷积、交叉熵与激活函数。
torch.set_num_threads(1)  # 固定小实验使用单线程以减少环境波动。
torch.manual_seed(41)  # 固定图像噪声、初始化与训练过程。
IMAGE_SIZE = 16  # 设置仓库俯视图边长为十六个网格。
CLASS_NAMES = {0: "纸箱", 1: "料桶"}  # 定义两个可读目标类别。
train_specs = [  # 定义十八张训练图的独立标注规格。
    [(0, 1, 1, 4, 4)],  # 单纸箱位于左上区域。
    [(1, 10, 1, 4, 4)],  # 单料桶位于右上区域。
    [(0, 1, 10, 4, 4)],  # 单纸箱位于左下区域。
    [(1, 10, 10, 4, 4)],  # 单料桶位于右下区域。
    [(0, 5, 5, 4, 4)],  # 单纸箱位于中心偏左。
    [(1, 7, 6, 4, 4)],  # 单料桶位于中心偏右。
    [(0, 1, 1, 4, 4), (1, 10, 10, 4, 4)],  # 两类目标分居主对角线。
    [(1, 1, 10, 4, 4), (0, 10, 1, 4, 4)],  # 两类目标分居副对角线。
    [(0, 1, 5, 4, 4), (1, 10, 5, 4, 4)],  # 两类目标水平分离。
    [(1, 5, 1, 4, 4), (0, 5, 10, 4, 4)],  # 两类目标垂直分离。
    [(0, 1, 1, 4, 4), (0, 10, 10, 4, 4)],  # 两个同类纸箱分离出现。
    [(1, 1, 10, 4, 4), (1, 10, 1, 4, 4)],  # 两个同类料桶分离出现。
    [(0, 2, 2, 3, 4), (1, 10, 9, 4, 3)],  # 使用不同长宽训练框回归。
    [(1, 2, 9, 3, 4), (0, 10, 2, 4, 3)],  # 交换类别并改变长宽。
    [(0, 1, 7, 4, 3)],  # 单纸箱使用扁平外形。
    [(1, 10, 7, 4, 3)],  # 单料桶使用扁平外形。
    [(0, 6, 1, 3, 4), (1, 6, 10, 3, 4)],  # 两目标在同一列但保持间隔。
    [(1, 1, 6, 4, 3), (0, 10, 6, 4, 3)],  # 两目标在同一行并交换类别。
]  # 完成训练图规格定义。
test_specs = [  # 定义六张未参与训练的新位置与尺寸组合。
    [(0, 3, 3, 4, 4)],  # 测试中间偏左上的单纸箱。
    [(1, 9, 3, 3, 4)],  # 测试中间偏右上的单料桶。
    [(0, 2, 2, 3, 4), (1, 9, 10, 4, 3)],  # 测试两类对角组合。
    [(1, 2, 10, 4, 3), (0, 10, 2, 3, 4)],  # 测试交换后的对角组合。
    [(0, 2, 6, 3, 3), (0, 10, 7, 4, 3)],  # 测试两个同类纸箱。
    [(1, 3, 2, 3, 4), (1, 9, 10, 4, 3)],  # 测试两个同类料桶。
]  # 完成留出图规格定义。
def render_scene(specification, seed):  # 根据可读物体规格渲染一张 RGB 网格图。
    generator = torch.Generator().manual_seed(seed)  # 为当前图像建立独立噪声源。
    image = torch.rand(3, IMAGE_SIZE, IMAGE_SIZE, generator=generator) * 0.035  # 创建带轻微噪声的黑色背景。
    targets = []  # 收集与图像分离保存的监督标注。
    for class_id, x0, y0, width, height in specification:  # 遍历当前图中的真实物体。
        if class_id == 0:  # 为纸箱绘制红黄相间纹理。
            image[0, y0:y0 + height, x0:x0 + width] = 0.88  # 写入纸箱主红色通道。
            image[1, y0:y0 + height, x0:x0 + width] = 0.28  # 写入纸箱辅绿色通道。
            image[2, y0:y0 + height, x0:x0 + width] = 0.10  # 写入纸箱较弱蓝色通道。
            image[1, y0:y0 + height:2, x0:x0 + width] = 0.55  # 添加可见水平封箱胶带纹理。
        else:  # 为料桶绘制蓝青相间纹理。
            image[0, y0:y0 + height, x0:x0 + width] = 0.10  # 写入料桶较弱红色通道。
            image[1, y0:y0 + height, x0:x0 + width] = 0.35  # 写入料桶辅绿色通道。
            image[2, y0:y0 + height, x0:x0 + width] = 0.90  # 写入料桶主蓝色通道。
            image[1, y0:y0 + height, x0:x0 + width:2] = 0.65  # 添加可见竖向桶箍纹理。
        center_x = (x0 + width / 2.0) / IMAGE_SIZE  # 把框中心横坐标归一化。
        center_y = (y0 + height / 2.0) / IMAGE_SIZE  # 把框中心纵坐标归一化。
        targets.append((class_id, center_x, center_y, width / IMAGE_SIZE, height / IMAGE_SIZE))  # 另存类别与中心宽高标注。
    return image, targets  # 返回观测图像和仅供监督使用的标注。
def build_dataset(specifications, seed_offset):  # 批量渲染一组图像与对应标注。
    images = []  # 收集模型可见的 RGB 图像。
    targets = []  # 收集仅供损失读取的目标集合。
    for index, specification in enumerate(specifications):  # 遍历场景规格并使用不同噪声种子。
        image, target = render_scene(specification, seed_offset + index)  # 渲染当前场景。
        images.append(image)  # 保存当前 RGB 观测。
        targets.append(target)  # 保存当前独立标注集合。
    return torch.stack(images), targets  # 返回批量图像和变长目标列表。
train_images, train_targets = build_dataset(train_specs, 1000)  # 构造十八张训练图像及监督。
test_images, test_targets = build_dataset(test_specs, 2000)  # 构造六张留出图像及监督。
def ascii_scene(image):  # 把小图转换成可读字符网格以直接检查输入。
    red_dominant = image[0] > image[2] + 0.25  # 从观测颜色识别红色前景位置。
    blue_dominant = image[2] > image[0] + 0.25  # 从观测颜色识别蓝色前景位置。
    rows = []  # 收集逐行字符表示。
    for row_index in range(0, IMAGE_SIZE, 2):  # 每两个像素采样一行以压缩显示。
        characters = []  # 收集当前显示行的字符。
        for column_index in range(0, IMAGE_SIZE, 2):  # 每两个像素采样一列以压缩显示。
            patch_red = bool(red_dominant[row_index:row_index + 2, column_index:column_index + 2].any())  # 判断小块是否包含纸箱颜色。
            patch_blue = bool(blue_dominant[row_index:row_index + 2, column_index:column_index + 2].any())  # 判断小块是否包含料桶颜色。
            characters.append("箱" if patch_red else "桶" if patch_blue else "·")  # 生成可读网格符号。
        rows.append("".join(characters))  # 保存当前压缩行。
    return "\n".join(rows)  # 合并为完整字符图。
print("训练/测试图像形状：", tuple(train_images.shape), tuple(test_images.shape))  # 展示真正进入模型的 RGB 张量尺寸。
print("留出图1的观测网格（不是标签 token）：\n" + ascii_scene(test_images[0]))  # 直接显示第一张测试图的像素语义。
print("留出图1独立标注：", test_targets[0])  # 单独展示 target 与 image 的数据边界。

训练/测试图像形状： (18, 3, 16, 16) (6, 3, 16, 16)
留出图1的观测网格（不是标签 token）：
········
·箱箱箱····
·箱箱箱····
·箱箱箱····
········
········
········
········
留出图1独立标注： [(0, 0.3125, 0.3125, 0.25, 0.25)]


## Baseline（基线）：整张前景合成一个检测框

基线只读取 RGB 像素：先用亮度阈值找前景，再对所有前景像素取一个外接框，类别由框内红蓝均值决定。它完全不知道真实目标数量，因此单目标通常可用，但多个分离目标会被错误合并。基线和 DETR 使用相同的逐图 precision、recall、匹配 IoU 指标。

In [2]:
def box_cxcywh_to_xyxy(boxes):  # 把中心宽高框转换成左上右下坐标。
    center_x, center_y, width, height = boxes.unbind(dim=-1)  # 拆分四个框分量。
    return torch.stack([center_x - width / 2.0, center_y - height / 2.0, center_x + width / 2.0, center_y + height / 2.0], dim=-1)  # 组合角点坐标。
def pairwise_iou(first_boxes, second_boxes):  # 手写两组框的两两 IoU 矩阵。
    first_xyxy = box_cxcywh_to_xyxy(first_boxes)  # 转换第一组框格式。
    second_xyxy = box_cxcywh_to_xyxy(second_boxes)  # 转换第二组框格式。
    left_top = torch.maximum(first_xyxy[:, None, :2], second_xyxy[None, :, :2])  # 计算交集左上角。
    right_bottom = torch.minimum(first_xyxy[:, None, 2:], second_xyxy[None, :, 2:])  # 计算交集右下角。
    intersection_size = (right_bottom - left_top).clamp_min(0.0)  # 截断无交集时的负边长。
    intersection = intersection_size[..., 0] * intersection_size[..., 1]  # 计算交集面积。
    first_area = (first_xyxy[:, 2] - first_xyxy[:, 0]).clamp_min(0.0) * (first_xyxy[:, 3] - first_xyxy[:, 1]).clamp_min(0.0)  # 计算第一组框面积。
    second_area = (second_xyxy[:, 2] - second_xyxy[:, 0]).clamp_min(0.0) * (second_xyxy[:, 3] - second_xyxy[:, 1]).clamp_min(0.0)  # 计算第二组框面积。
    union = first_area[:, None] + second_area[None, :] - intersection  # 计算并集面积。
    return intersection / union.clamp_min(1e-8)  # 返回数值稳定的 IoU。
def target_tensors(target):  # 把变长标注列表转换为类别与框张量。
    classes = torch.tensor([item[0] for item in target], dtype=torch.long)  # 提取真实类别编号。
    boxes = torch.tensor([item[1:] for item in target], dtype=torch.float32)  # 提取归一化中心宽高框。
    return classes, boxes  # 返回集合监督张量。
def foreground_baseline(image):  # 仅从图像观测生成至多一个基线检测结果。
    foreground = image.max(dim=0).values > 0.45  # 用固定亮度阈值寻找可见前景。
    coordinates = foreground.nonzero(as_tuple=False)  # 获取全部前景像素坐标。
    if len(coordinates) == 0:  # 处理完全没有前景的边界情况。
        return []  # 无前景时不输出任何框。
    y_min = int(coordinates[:, 0].min())  # 读取前景最小纵坐标。
    y_max = int(coordinates[:, 0].max()) + 1  # 读取前景最大纵坐标的开区间端点。
    x_min = int(coordinates[:, 1].min())  # 读取前景最小横坐标。
    x_max = int(coordinates[:, 1].max()) + 1  # 读取前景最大横坐标的开区间端点。
    region_mean = image[:, foreground].mean(dim=1)  # 计算全部前景的平均 RGB 观测。
    class_id = 0 if float(region_mean[0]) >= float(region_mean[2]) else 1  # 用主颜色决定纸箱或料桶。
    box = torch.tensor([(x_min + x_max) / (2.0 * IMAGE_SIZE), (y_min + y_max) / (2.0 * IMAGE_SIZE), (x_max - x_min) / IMAGE_SIZE, (y_max - y_min) / IMAGE_SIZE])  # 构造单个归一化外接框。
    return [(class_id, box, 1.0)]  # 返回不依赖真实数量的一个检测结果。
def score_detections(predictions_by_image, targets_by_image):  # 用一对一匹配统计检测 precision、recall 与 IoU。
    true_positive = 0  # 初始化正确检测数量。
    false_positive = 0  # 初始化多报或错误检测数量。
    false_negative = 0  # 初始化漏报目标数量。
    matched_ious = []  # 收集每个真实目标对应的最佳匹配 IoU。
    rows = []  # 收集逐图可读评估结果。
    for image_index, (predictions, target) in enumerate(zip(predictions_by_image, targets_by_image), start=1):  # 对齐每张图的预测与标注。
        target_classes, target_boxes = target_tensors(target)  # 读取当前图真实集合。
        prediction_classes = torch.tensor([item[0] for item in predictions], dtype=torch.long) if predictions else torch.empty(0, dtype=torch.long)  # 构造预测类别张量。
        prediction_boxes = torch.stack([item[1] for item in predictions]) if predictions else torch.empty(0, 4)  # 构造预测框张量。
        pair_count = min(len(predictions), len(target))  # 计算最多可形成的一一匹配数量。
        best_pairs = []  # 初始化当前图的最佳匹配对。
        best_iou_sum = -1.0  # 初始化匹配总 IoU。
        if pair_count > 0:  # 只有预测和真实均非空时才搜索匹配。
            iou_matrix = pairwise_iou(prediction_boxes, target_boxes)  # 计算全部预测与真实框 IoU。
            for prediction_indices in itertools.permutations(range(len(predictions)), pair_count):  # 枚举参与匹配的预测索引排列。
                for target_indices in itertools.permutations(range(len(target)), pair_count):  # 枚举参与匹配的目标索引排列。
                    current_sum = sum(float(iou_matrix[prediction_index, target_index]) for prediction_index, target_index in zip(prediction_indices, target_indices))  # 汇总当前一一对应的 IoU。
                    if current_sum > best_iou_sum:  # 保留总 IoU 最大的集合对应。
                        best_iou_sum = current_sum  # 更新最佳 IoU 总和。
                        best_pairs = list(zip(prediction_indices, target_indices))  # 保存最佳预测目标索引对。
        image_true_positive = 0  # 初始化当前图正确检测数。
        for prediction_index, target_index in best_pairs:  # 检查最佳匹配中的类别与定位门槛。
            current_iou = float(pairwise_iou(prediction_boxes[prediction_index:prediction_index + 1], target_boxes[target_index:target_index + 1])[0, 0])  # 读取当前匹配 IoU。
            matched_ious.append(current_iou)  # 保存定位质量用于平均。
            class_correct = int(prediction_classes[prediction_index]) == int(target_classes[target_index])  # 判断匹配类别是否正确。
            image_true_positive += int(class_correct and current_iou >= 0.5)  # 同时满足类别和 IoU 才计为正确检测。
        true_positive += image_true_positive  # 累加全局正确检测数。
        false_positive += len(predictions) - image_true_positive  # 未成为正确检测的输出均计为误报。
        false_negative += len(target) - image_true_positive  # 未被正确命中的目标均计为漏报。
        rows.append((image_index, len(target), len(predictions), image_true_positive, best_iou_sum / max(pair_count, 1) if pair_count else 0.0))  # 保存逐图数量与平均匹配 IoU。
    precision = true_positive / max(true_positive + false_positive, 1)  # 计算检测精确率。
    recall = true_positive / max(true_positive + false_negative, 1)  # 计算检测召回率。
    mean_iou = sum(matched_ious) / max(len(matched_ious), 1)  # 计算所有匹配对平均 IoU。
    return precision, recall, mean_iou, rows  # 返回汇总指标与逐图账本。
baseline_predictions = [foreground_baseline(image) for image in test_images]  # 在六张留出图上执行无 oracle 数量基线。
baseline_metrics = score_detections(baseline_predictions, test_targets)  # 计算基线检测指标。
print("图  gold数  预测数  TP  匹配IoU")  # 输出逐图基线结果表头。
for row in baseline_metrics[3]:  # 遍历六张图的基线账本。
    print(f"{row[0]:>2}    {row[1]:>2}      {row[2]:>2}    {row[3]:>2}    {row[4]:.3f}")  # 展示基线合并多目标造成的具体错误。
print(f"Baseline precision={baseline_metrics[0]:.3f}，recall={baseline_metrics[1]:.3f}，匹配IoU={baseline_metrics[2]:.3f}")  # 汇总同口径基线指标。

图  gold数  预测数  TP  匹配IoU
 1     1       1     1    1.000
 2     1       1     1    1.000
 3     2       1     0    0.099
 4     2       1     0    0.099
 5     2       1     0    0.250
 6     2       1     0    0.109
Baseline precision=0.333，recall=0.200，匹配IoU=0.426


## 核心实现一：从 RGB 网格提取 token，并手写 query cross-attention

每个 token 来自一个 2×2 图像 patch 的 RGB 均值和二维坐标，共 5 个观测字段。这里没有候选框、类别 one-hot 或 objectness。三个可学习 query 与 64 个视觉 token 做缩放点积注意力，随后各自输出三分类 logits（纸箱、料桶、`no-object`）和一个归一化框。

In [3]:
class ManualDETR(nn.Module):  # 定义只读取图像观测的轻量 DETR。
    def __init__(self, hidden_dim=40, query_count=3, class_count=3):  # 初始化 token、query、注意力与检测头。
        super().__init__()  # 注册基础模块状态。
        self.hidden_dim = hidden_dim  # 保存隐藏维度用于缩放点积。
        self.query_count = query_count  # 保存固定 query 数量。
        coordinate_values = (torch.arange(8, dtype=torch.float32) + 0.5) / 8.0  # 计算八乘八 patch 中心坐标。
        grid_y, grid_x = torch.meshgrid(coordinate_values, coordinate_values, indexing="ij")  # 生成二维 patch 坐标网格。
        coordinates = torch.stack([grid_x.reshape(-1), grid_y.reshape(-1)], dim=-1)  # 整理为六十四个二维位置。
        self.register_buffer("patch_coordinates", coordinates)  # 保存不可学习且与图像内容独立的位置编码。
        self.token_weight = nn.Parameter(torch.randn(5, hidden_dim) * 0.20)  # 创建五维观测到视觉 token 的投影。
        self.token_bias = nn.Parameter(torch.zeros(hidden_dim))  # 创建视觉 token 偏置。
        self.query_embeddings = nn.Parameter(torch.randn(query_count, hidden_dim) * 0.30)  # 创建三个可学习 object query。
        self.query_weight = nn.Parameter(torch.randn(hidden_dim, hidden_dim) / math.sqrt(hidden_dim))  # 创建 query 投影矩阵。
        self.key_weight = nn.Parameter(torch.randn(hidden_dim, hidden_dim) / math.sqrt(hidden_dim))  # 创建视觉 key 投影矩阵。
        self.value_weight = nn.Parameter(torch.randn(hidden_dim, hidden_dim) / math.sqrt(hidden_dim))  # 创建视觉 value 投影矩阵。
        self.context_weight = nn.Parameter(torch.randn(hidden_dim, hidden_dim) / math.sqrt(hidden_dim))  # 创建注意力上下文融合矩阵。
        self.class_weight = nn.Parameter(torch.randn(hidden_dim, class_count) * 0.16)  # 创建含 no-object 的分类头。
        self.class_bias = nn.Parameter(torch.zeros(class_count))  # 创建分类头偏置。
        self.box_weight = nn.Parameter(torch.randn(hidden_dim, 4) * 0.16)  # 创建中心宽高回归头。
        self.box_bias = nn.Parameter(torch.zeros(4))  # 创建框回归偏置。
    def image_tokens(self, images):  # 从 RGB 像素计算二乘二 patch 观测 token。
        patches = images.unfold(2, 2, 2).unfold(3, 2, 2)  # 无参数地切分为八乘八个 patch。
        rgb_means = patches.mean(dim=(-1, -2)).permute(0, 2, 3, 1).reshape(images.shape[0], 64, 3)  # 计算每块 RGB 均值。
        coordinates = self.patch_coordinates.unsqueeze(0).expand(images.shape[0], -1, -1)  # 为每张图复制相同位置坐标。
        observed_features = torch.cat([rgb_means, coordinates], dim=-1)  # 拼接图像统计和位置而不接触 target。
        tokens = torch.tanh(observed_features @ self.token_weight + self.token_bias)  # 映射为视觉隐藏表示。
        return tokens, observed_features  # 返回视觉 token 与可审计的原始观测字段。
    def forward(self, images, return_details=False):  # 执行视觉编码、query 注意力与集合预测。
        tokens, observed_features = self.image_tokens(images)  # 只从输入图像生成视觉 token。
        queries = self.query_embeddings.unsqueeze(0).expand(images.shape[0], -1, -1)  # 为 batch 复制三个可学习 query。
        projected_queries = queries @ self.query_weight  # 投影 object query。
        projected_keys = tokens @ self.key_weight  # 投影视觉 key。
        projected_values = tokens @ self.value_weight  # 投影视觉 value。
        attention_scores = torch.einsum("bqd,bnd->bqn", projected_queries, projected_keys) / math.sqrt(self.hidden_dim)  # 手写 query-token 缩放点积。
        foreground_hint = observed_features[..., :3].max(dim=-1).values  # 从 RGB 亮度计算非监督的前景提示。
        attention_scores = attention_scores + 1.5 * foreground_hint.unsqueeze(1)  # 让初期 query 更容易看到真实像素前景。
        attention = torch.softmax(attention_scores, dim=-1)  # 沿六十四个视觉位置归一化注意力。
        context = torch.einsum("bqn,bnd->bqd", attention, projected_values)  # 聚合每个 query 的图像证据。
        decoded = torch.tanh(queries + context @ self.context_weight)  # 融合 query 身份与视觉上下文。
        class_logits = decoded @ self.class_weight + self.class_bias  # 输出纸箱、料桶和 no-object 分数。
        boxes = torch.sigmoid(decoded @ self.box_weight + self.box_bias)  # 输出归一化中心宽高框。
        if return_details:  # 教学模式需要展示输入与注意力中间量。
            return class_logits, boxes, attention, observed_features  # 返回完整可审计中间结果。
        return class_logits, boxes  # 训练模式返回集合预测。
torch.manual_seed(43)  # 固定 DETR 参数初始化。
detr = ManualDETR()  # 创建三个 query 的手写检测模型。
preview_logits, preview_boxes, preview_attention, preview_features = detr(test_images[:2], return_details=True)  # 对两张留出图执行未训练前向。
print("图像→观测特征→attention→输出形状：", tuple(test_images[:2].shape), tuple(preview_features.shape), tuple(preview_attention.shape), tuple(preview_logits.shape), tuple(preview_boxes.shape))  # 展示完整张量路径。
print("首个 patch 的 [R,G,B,x,y]：", [round(float(value), 3) for value in preview_features[0, 0]])  # 证明 token 字段只有图像与坐标。
print("query1 注意力最高的五个 patch：", torch.topk(preview_attention[0, 0], 5).indices.tolist())  # 展示未训练 query 实际读取的位置。

图像→观测特征→attention→输出形状： (2, 3, 16, 16) (2, 64, 5) (2, 3, 64) (2, 3, 3) (2, 3, 4)
首个 patch 的 [R,G,B,x,y]： [0.013, 0.017, 0.016, 0.062, 0.062]
query1 注意力最高的五个 patch： [18, 17, 26, 19, 10]


## 核心实现二：穷举小规模 Hungarian 等价匹配与集合损失

三个 query、最多两个目标时，可穷举 query 排列得到与 Hungarian 相同的最小一一匹配。匹配代价结合类别负对数概率、框 L1 和 `1-IoU`，索引选择使用 `detach`，真正 loss 仍从原 logits/box 反向传播。未匹配 query 的类别目标统一为 `no-object=2`。

In [4]:
def choose_query_assignment(class_logits, predicted_boxes, target_classes, target_boxes):  # 为单张图寻找最小代价 query-target 匹配。
    class_cost = -torch.log_softmax(class_logits, dim=-1)[:, target_classes]  # 计算每个 query 对每个真实类别的负 log 概率。
    l1_cost = torch.cdist(predicted_boxes, target_boxes, p=1)  # 计算所有预测框与真实框的 L1 距离。
    iou_cost = 1.0 - pairwise_iou(predicted_boxes, target_boxes)  # 计算定位不重合代价。
    total_cost = class_cost + 4.0 * l1_cost + 2.0 * iou_cost  # 合并分类与定位匹配代价。
    best_queries = None  # 初始化最佳 query 排列。
    best_cost = float("inf")  # 初始化最小集合代价。
    for query_indices in itertools.permutations(range(class_logits.shape[0]), len(target_classes)):  # 枚举不重复 query 到目标的对应。
        current_cost = sum(float(total_cost[query_index, target_index].detach()) for target_index, query_index in enumerate(query_indices))  # 汇总当前匹配代价且不对离散选择求导。
        if current_cost < best_cost:  # 更新目前最优的一一对应。
            best_cost = current_cost  # 保存更小集合代价。
            best_queries = query_indices  # 保存对应 query 索引。
    return list(enumerate(best_queries)), total_cost  # 返回目标索引到 query 索引的匹配与完整代价矩阵。
def detr_set_loss(class_logits_batch, predicted_boxes_batch, targets):  # 计算一个 batch 的 DETR 集合损失。
    classification_losses = []  # 收集逐图分类损失。
    box_losses = []  # 收集逐图框坐标损失。
    iou_losses = []  # 收集逐图 IoU 损失。
    assignments = []  # 收集匹配索引用于解释。
    class_weights = torch.tensor([1.0, 1.0, 0.25], device=class_logits_batch.device)  # 降低大量 no-object 对分类的支配。
    for image_index, target in enumerate(targets):  # 遍历 batch 中每张图的变长目标集合。
        target_classes, target_boxes = target_tensors(target)  # 转换当前目标集合。
        target_classes = target_classes.to(class_logits_batch.device)  # 把类别移到模型设备。
        target_boxes = target_boxes.to(predicted_boxes_batch.device)  # 把框移到模型设备。
        matched_pairs, _ = choose_query_assignment(class_logits_batch[image_index], predicted_boxes_batch[image_index], target_classes, target_boxes)  # 根据当前预测选择集合匹配。
        query_class_targets = torch.full((class_logits_batch.shape[1],), 2, dtype=torch.long, device=class_logits_batch.device)  # 默认所有 query 都监督为 no-object。
        matched_query_indices = []  # 收集真正负责物体的 query 索引。
        matched_target_indices = []  # 收集与 query 对齐的目标索引。
        for target_index, query_index in matched_pairs:  # 写入当前最优匹配的类别监督。
            query_class_targets[query_index] = target_classes[target_index]  # 把匹配 query 的目标从 no-object 改为真实类别。
            matched_query_indices.append(query_index)  # 保存匹配 query 索引。
            matched_target_indices.append(target_index)  # 保存对应目标索引。
        classification_losses.append(F.cross_entropy(class_logits_batch[image_index], query_class_targets, weight=class_weights))  # 计算含 no-object 的 query 分类损失。
        matched_prediction_boxes = predicted_boxes_batch[image_index, matched_query_indices]  # 读取匹配 query 的预测框。
        matched_target_boxes = target_boxes[matched_target_indices]  # 按相同顺序读取真实框。
        box_losses.append(F.l1_loss(matched_prediction_boxes, matched_target_boxes))  # 计算匹配框 L1 损失。
        diagonal_ious = pairwise_iou(matched_prediction_boxes, matched_target_boxes).diagonal()  # 读取一一对应匹配的 IoU。
        iou_losses.append((1.0 - diagonal_ious).mean())  # 鼓励预测框与真实框重合。
        assignments.append(matched_pairs)  # 保存本轮匹配结果。
    classification_loss = torch.stack(classification_losses).mean()  # 汇总 batch 分类损失。
    box_loss = torch.stack(box_losses).mean()  # 汇总 batch 框 L1 损失。
    iou_loss = torch.stack(iou_losses).mean()  # 汇总 batch IoU 损失。
    total_loss = classification_loss + 5.0 * box_loss + 2.0 * iou_loss  # 形成最终可反向传播集合目标。
    return total_loss, classification_loss, box_loss, iou_loss, assignments  # 返回损失分项与匹配证据。
initial_logits, initial_boxes = detr(train_images[:2])  # 对两张训练图生成初始集合预测。
initial_loss_parts = detr_set_loss(initial_logits, initial_boxes, train_targets[:2])  # 计算未训练集合损失。
print("首两图初始匹配：", initial_loss_parts[4])  # 展示 query 编号不是固定绑定标注顺序。
print("初始 loss 分项：", [round(float(value.detach()), 4) for value in initial_loss_parts[:4]])  # 展示总损失、分类、L1 与 IoU 中间量。

首两图初始匹配： [[(0, 1)], [(0, 0)]]
初始 loss 分项： [4.1455, 1.2195, 0.2173, 0.9197]


## 真实训练、无 oracle 推理与逐图结果

训练只在 18 张图上最小化集合损失。推理时对每个 query 直接取三分类 argmax，预测为 `no-object` 的 query 丢弃；因此输出数量由模型自身决定。六张测试图使用新的位置/尺寸组合，但仍来自同一合成渲染规则，指标只能说明本教学分布内的定位与集合机制。

In [5]:
optimizer = torch.optim.Adam(detr.parameters(), lr=0.008)  # 创建更新所有手写 DETR 参数的 Adam。
training_history = []  # 保存 loss 分项、梯度和训练检测数。
for epoch in range(900):  # 在十八张图上执行真实集合预测训练。
    optimizer.zero_grad()  # 清除上一轮累计梯度。
    train_logits, train_boxes = detr(train_images)  # 从 RGB 图像得到三个 query 的类别与框。
    total_loss, classification_loss, box_loss, iou_loss, train_assignments = detr_set_loss(train_logits, train_boxes, train_targets)  # 执行动态集合匹配并计算损失。
    total_loss.backward()  # 把分类与定位误差传播到 query 和视觉投影。
    query_gradient = float(detr.query_embeddings.grad.norm().detach())  # 记录 object query 的真实梯度。
    torch.nn.utils.clip_grad_norm_(detr.parameters(), 5.0)  # 限制小数据训练中的极端梯度。
    optimizer.step()  # 根据当前集合目标更新参数。
    predicted_object_count = int((train_logits.argmax(dim=-1) != 2).sum())  # 统计模型自己预测的训练目标数量。
    training_history.append((float(total_loss.detach()), float(classification_loss.detach()), float(box_loss.detach()), float(iou_loss.detach()), query_gradient, predicted_object_count))  # 保存完整训练诊断。
def predictions_from_model(model, images):  # 在不知道 gold 数量时把 query 输出转换为检测列表。
    with torch.no_grad():  # 关闭评估阶段梯度。
        logits, boxes, attention, observed_features = model(images, return_details=True)  # 执行模型前向并保留注意力证据。
        probabilities = torch.softmax(logits, dim=-1)  # 计算三个类别概率。
    predictions_by_image = []  # 收集逐图变长检测列表。
    for image_index in range(images.shape[0]):  # 遍历每张待检测图像。
        predictions = []  # 初始化当前图检测结果。
        for query_index in range(model.query_count):  # 检查三个 query 的自主类别决策。
            class_id = int(probabilities[image_index, query_index].argmax())  # 选择当前 query 概率最高类别。
            confidence = float(probabilities[image_index, query_index, class_id])  # 读取对应置信度用于展示。
            if class_id != 2:  # 只有非 no-object query 才输出检测。
                predictions.append((class_id, boxes[image_index, query_index].detach(), confidence))  # 保存类别、框与置信度。
        predictions_by_image.append(predictions)  # 保存当前图的自主数量预测。
    return predictions_by_image, probabilities, boxes, attention  # 返回检测集合与中间张量。
detr_predictions, detr_probabilities, detr_boxes, detr_attention = predictions_from_model(detr, test_images)  # 在六张留出图上执行无 oracle 推理。
detr_metrics = score_detections(detr_predictions, test_targets)  # 使用与基线相同口径评估 DETR。
print("epoch  total    cls      L1      IoUloss  query梯度  训练预测数")  # 输出训练过程表头。
for epoch in (0, 99, 399, 899):  # 选择四个关键训练阶段。
    row = training_history[epoch]  # 读取当前阶段诊断。
    print(f"{epoch + 1:>4}  {row[0]:>6.3f}  {row[1]:>6.3f}  {row[2]:>6.3f}  {row[3]:>7.3f}   {row[4]:>7.4f}       {row[5]:>2}")  # 展示损失与 query 梯度变化。
print("图  gold数/预测数  预测类别(置信度)       TP  匹配IoU")  # 输出逐图留出检测结果表头。
for image_index, row in enumerate(detr_metrics[3]):  # 对齐检测列表与逐图指标。
    prediction_text = [f"{CLASS_NAMES[item[0]]}:{item[2]:.2f}" for item in detr_predictions[image_index]]  # 格式化模型自主输出类别和置信度。
    print(f"{image_index + 1:>2}      {row[1]}/{row[2]}       {str(prediction_text):<27}  {row[3]:>2}    {row[4]:.3f}")  # 展示每张图的数量、类别和定位质量。
print(f"同口径指标：Baseline P/R/IoU={baseline_metrics[0]:.3f}/{baseline_metrics[1]:.3f}/{baseline_metrics[2]:.3f}；DETR P/R/IoU={detr_metrics[0]:.3f}/{detr_metrics[1]:.3f}/{detr_metrics[2]:.3f}")  # 诚实汇总基线与模型结果。
print("测试图3各 query 注意力峰值 patch：", [int(detr_attention[2, query_index].argmax()) for query_index in range(detr.query_count)])  # 展示不同 query 实际聚焦位置。

epoch  total    cls      L1      IoUloss  query梯度  训练预测数
   1   3.796   1.072   0.191    0.885    0.4096       36
 100   1.165   0.139   0.032    0.433    0.7718       32
 400   0.583   0.019   0.017    0.239    1.2579       29
 900   0.349   0.003   0.010    0.148    0.5966       28
图  gold数/预测数  预测类别(置信度)       TP  匹配IoU
 1      1/1       ['纸箱:0.99']                   0    0.437
 2      1/1       ['料桶:0.66']                   0    0.224
 3      2/2       ['纸箱:1.00', '料桶:1.00']        1    0.474
 4      2/2       ['纸箱:1.00', '料桶:1.00']        2    0.627
 5      2/2       ['纸箱:0.99', '纸箱:1.00']        0    0.239
 6      2/1       ['料桶:1.00']                   0    0.163
同口径指标：Baseline P/R/IoU=0.333/0.200/0.426；DETR P/R/IoU=0.333/0.300/0.389
测试图3各 query 注意力峰值 patch： [9, 9, 45]


## 结果解读

结果表必须同时看“预测数量、类别和 IoU”，不能只看平均 loss。保存输出中，DETR 的 recall 从基线 0.20 提高到 0.30，但匹配 IoU 从 0.426 降到 0.389，且同类双目标仍出现漏报；这说明训练 loss 下降不等于留出定位更好。模型确实通过 `no-object` 自己决定输出数量，但三个 query 的注意力仍有重叠，小数据 DETR 未必胜过带强视觉先验的规则。这里的留出图仍共享颜色和渲染机制，绝不能称为真实仓库泛化。

## 失败案例：固定 query 序号绑定标注顺序

目标检测标签本质上是集合。同一张图把两个标注交换顺序后，固定槽位 L1 loss 会改变；集合匹配 loss 应保持不变。下面直接用训练后预测复现这种顺序敏感错误。

In [6]:
probe_logits = detr(test_images[2:3])[0][0].detach()  # 读取含两个目标测试图的 query 分类输出。
probe_boxes = detr(test_images[2:3])[1][0].detach()  # 读取同一图的 query 框输出。
probe_classes, probe_targets = target_tensors(test_targets[2])  # 读取原始目标顺序。
swapped_classes = probe_classes.flip(0)  # 交换两个目标类别顺序但不改变集合。
swapped_targets = probe_targets.flip(0)  # 同步交换两个目标框顺序。
fixed_original = F.l1_loss(probe_boxes[:2], probe_targets)  # 错误地把前两个 query 固定绑定原标注顺序。
fixed_swapped = F.l1_loss(probe_boxes[:2], swapped_targets)  # 用交换标注计算同一固定槽位损失。
matched_original, original_cost = choose_query_assignment(probe_logits, probe_boxes, probe_classes, probe_targets)  # 对原始顺序执行集合匹配。
matched_swapped, swapped_cost = choose_query_assignment(probe_logits, probe_boxes, swapped_classes, swapped_targets)  # 对交换顺序执行集合匹配。
set_original = sum(float(original_cost[query_index, target_index]) for target_index, query_index in matched_original)  # 汇总原始标注的最优集合代价。
set_swapped = sum(float(swapped_cost[query_index, target_index]) for target_index, query_index in matched_swapped)  # 汇总交换标注的最优集合代价。
print(f"固定槽位 L1：原顺序={float(fixed_original):.4f}，交换顺序={float(fixed_swapped):.4f}")  # 展示错误方案依赖标注排列。
print("集合匹配：原顺序", matched_original, "交换顺序", matched_swapped)  # 展示匹配会自动调整 query 对应。
print(f"集合最优代价：原顺序={set_original:.6f}，交换顺序={set_swapped:.6f}")  # 验证正确集合目标对排列保持不变。

固定槽位 L1：原顺序=0.2092，交换顺序=0.0862
集合匹配：原顺序 [(0, 1), (1, 2)] 交换顺序 [(0, 2), (1, 1)]
集合最优代价：原顺序=3.018154，交换顺序=3.018154


## 生产差距与追问

真实 DETR 还需要大规模视觉 backbone、多尺度特征、遮挡与密集小目标、数据增强、类别长尾、GIoU、真正 Hungarian 算法、置信度校准和 COCO AP。输入管线必须做数据血缘审计，严禁把标注框、类别或由标注直接计算的 objectness 混入模型特征；候选框若来自独立 detector，也要明确其训练数据和线上可用性。本实验只有低分辨率合成颜色块，重点是证明“图像观测 → query → matching/loss”的合同，而不是宣称达到真实检测质量。

## 最小回归测试

In [7]:
assert len(test_images) >= 5  # 保证留出案例足以形成逐图对照。
assert preview_features.shape[-1] == 5  # 保护 token 只含 RGB 均值和二维坐标五个观测字段。
assert all(len(predictions) <= detr.query_count for predictions in detr_predictions)  # 保护推理数量完全来自固定 query 而非 gold 数量。
assert abs(set_original - set_swapped) < 1e-5  # 保护集合匹配对目标排列保持不变。
assert abs(float(fixed_original) - float(fixed_swapped)) > 1e-4  # 保护失败案例确实暴露固定槽位顺序敏感性。
assert torch.isfinite(torch.tensor(detr_metrics[:3])).all()  # 保护测试检测指标均为有限数值。
print("最小回归测试通过：图像输入边界、无 oracle 推理与集合匹配不变量均保持有效。")  # 输出集中测试结论。

最小回归测试通过：图像输入边界、无 oracle 推理与集合匹配不变量均保持有效。
